# KernelSHAP vs GradientSHAP vs DeepLIFT SHAP
## Accuracy vs. Speed Trade-off Benchmarks

**Learning objectives:**
- Run all three SHAP methods on the same model and data
- Measure computation time for each method
- Quantify attribution agreement using rank correlation
- Identify which method is most appropriate for different scenarios

**Estimated time:** 15 minutes

**Model:** Pretrained MLP on tabular data (UCI Adult Income)

**Methods compared:** `KernelShap`, `GradientShap`, `DeepLiftShap`

In [ ]:
learning_objectives(["Speed Trade-off Benchmarks", "- Run all three SHAP methods on the same model and data", "Measure computation time for each method", "Quantify attribution agreement using rank correlation", "Identify which method is most appropriate for different scenarios", "15 minutes"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# Captum attribution methods
from captum.attr import KernelShap, GradientShap, DeepLiftShap

print("All imports successful.")
print(f"PyTorch version: {torch.__version__}")

## 1. Load and Preprocess Real Data

We use the UCI Adult Income dataset — a standard tabular benchmark with 14 features predicting whether income exceeds $50K/year.

In [ ]:
# Load UCI Adult Income dataset
adult = fetch_openml('adult', version=2, as_frame=True, parser='auto')
df = adult.frame.copy()

# Select a clean subset of features
feature_cols = ['age', 'education-num', 'hours-per-week', 'capital-gain',
                'capital-loss', 'fnlwgt', 'workclass', 'marital-status',
                'occupation', 'relationship', 'race', 'sex', 'native-country']

target_col = 'class'

# Encode categoricals
df_processed = df[feature_cols + [target_col]].dropna()
for col in df_processed.select_dtypes(include='category').columns:
    df_processed[col] = LabelEncoder().fit_transform(df_processed[col].astype(str))

X = df_processed[feature_cols].values.astype(np.float32)
y = (df_processed[target_col].astype(str).str.strip() == '>50K').astype(np.float32).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")
print(f"Features:         {X_train.shape[1]}")
print(f"Class balance:    {y_train.mean():.1%} positive")

## 2. Train a Tabular MLP

We train a 3-layer MLP on the Adult dataset. This gives us a real model whose predictions we'll explain.

In [ ]:
class TabularMLP(nn.Module):
    """3-layer MLP for tabular binary classification."""
    def __init__(self, input_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, 2),  # 2 classes for Captum compatibility
        )

    def forward(self, x):
        return self.net(x)


# Convert to tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

# Train
model = TabularMLP(input_dim=X_train.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(30):
    optimizer.zero_grad()
    logits = model(X_train_t)
    loss = criterion(logits, y_train_t)
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    preds = model(X_test_t).argmax(dim=1)
    accuracy = (preds == y_test_t).float().mean().item()
print(f"Test accuracy: {accuracy:.1%}")

## 3. Prepare Background and Test Inputs

All SHAP methods need a background distribution. We use 100 training samples.

In [ ]:
# Background: 100 random training samples
np.random.seed(42)
bg_idx = np.random.choice(len(X_train), 100, replace=False)
background = X_train_t[bg_idx]  # shape: (100, 13)

# Select 5 test inputs to explain (diverse predictions)
with torch.no_grad():
    test_probs = torch.softmax(model(X_test_t), dim=1)[:, 1]

# Pick inputs spread across prediction range
sorted_idx = test_probs.argsort()
test_indices = [
    sorted_idx[10].item(),   # low probability
    sorted_idx[25].item(),
    sorted_idx[len(sorted_idx)//2].item(),  # near decision boundary
    sorted_idx[-25].item(),
    sorted_idx[-10].item(),  # high probability
]

test_inputs = X_test_t[test_indices]  # shape: (5, 13)
test_targets = y_test_t[test_indices]

print("Selected test inputs:")
for i, idx in enumerate(test_indices):
    prob = test_probs[idx].item()
    print(f"  Input {i}: P(>50K) = {prob:.3f}")

## 4. KernelSHAP — Timing Benchmark

KernelSHAP is theoretically optimal but slow. We measure time for different `n_samples` values.

In [ ]:
kernel_shap = KernelShap(model)

ks_times = {}
ks_attributions = {}

for n_samples in [50, 100, 200, 500]:
    start = time.perf_counter()
    attrs = kernel_shap.attribute(
        inputs=test_inputs,
        baselines=background,
        n_samples=n_samples,
        perturbations_per_eval=64,
        target=1,  # explain class 1 (>50K)
    )
    elapsed = time.perf_counter() - start
    ks_times[n_samples] = elapsed
    ks_attributions[n_samples] = attrs.detach()
    print(f"KernelSHAP n_samples={n_samples:4d}: {elapsed:.2f}s")

# Use n_samples=500 as our "reference" KernelSHAP
ks_reference = ks_attributions[500]

## 5. GradientSHAP — Timing Benchmark

In [ ]:
grad_shap = GradientShap(model)

gs_times = {}
gs_attributions = {}

for n_samples in [10, 25, 50, 100]:
    start = time.perf_counter()
    attrs, delta = grad_shap.attribute(
        inputs=test_inputs,
        baselines=background,
        n_samples=n_samples,
        target=1,
        return_convergence_delta=True,
    )
    elapsed = time.perf_counter() - start
    gs_times[n_samples] = elapsed
    gs_attributions[n_samples] = attrs.detach()
    print(f"GradientSHAP n_samples={n_samples:3d}: {elapsed:.2f}s  |  max |δ|: {delta.abs().max().item():.4f}")

gs_reference = gs_attributions[50]

## 6. DeepLIFT SHAP — Timing Benchmark

In [ ]:
dl_shap = DeepLiftShap(model)

dl_times = {}
dl_attributions = {}

for n_bg in [10, 25, 50, 100]:
    bg_subset = background[:n_bg]
    start = time.perf_counter()
    attrs, delta = dl_shap.attribute(
        inputs=test_inputs,
        baselines=bg_subset,
        target=1,
        return_convergence_delta=True,
    )
    elapsed = time.perf_counter() - start
    dl_times[n_bg] = elapsed
    dl_attributions[n_bg] = attrs.detach()
    print(f"DeepLIFT SHAP n_bg={n_bg:3d}: {elapsed:.2f}s  |  max |δ|: {delta.abs().max().item():.6f}")

dl_reference = dl_attributions[50]

## 7. Speed Comparison Plot

Visualize the computation time for each method across different sample counts.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Absolute time
ax = axes[0]
ax.plot(list(ks_times.keys()), list(ks_times.values()), 'o-', label='KernelSHAP', color='#e41a1c', linewidth=2)
ax.plot(list(gs_times.keys()), list(gs_times.values()), 's-', label='GradientSHAP', color='#377eb8', linewidth=2)
ax.plot(list(dl_times.keys()), list(dl_times.values()), '^-', label='DeepLIFT SHAP', color='#4daf4a', linewidth=2)
ax.set_xlabel('Number of samples / baselines')
ax.set_ylabel('Time (seconds)')
ax.set_title('Computation Time vs. Sample Count')
ax.legend()
ax.grid(alpha=0.3)

# Right: Method comparison at comparable quality
ax = axes[1]
methods = ['KernelSHAP\n(n=200)', 'GradientSHAP\n(n=50)', 'DeepLIFT SHAP\n(n=50)']
times = [ks_times[200], gs_times[50], dl_times[50]]
colors = ['#e41a1c', '#377eb8', '#4daf4a']
bars = ax.bar(methods, times, color=colors, edgecolor='white', linewidth=1.5)
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{t:.2f}s', ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Time (seconds)')
ax.set_title('Speed Comparison at Practical Settings')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('SHAP Methods: Computation Time Comparison\n(5 test inputs, 13 features, adult income dataset)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_timing_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: shap_timing_comparison.png")

## 8. Attribution Agreement Analysis

Speed only matters if the methods agree on *which features are important*. We use Spearman rank correlation to measure attribution agreement.

In [ ]:
feature_names = feature_cols
n_inputs = test_inputs.shape[0]

# Compute pairwise Spearman correlations for each test input
corr_ks_gs = []
corr_ks_dl = []
corr_gs_dl = []

for i in range(n_inputs):
    a_ks = ks_reference[i].numpy()
    a_gs = gs_reference[i].numpy()
    a_dl = dl_reference[i].numpy()

    corr_ks_gs.append(spearmanr(a_ks, a_gs).correlation)
    corr_ks_dl.append(spearmanr(a_ks, a_dl).correlation)
    corr_gs_dl.append(spearmanr(a_gs, a_dl).correlation)

print("Spearman Rank Correlation (attribution agreement):")
print(f"  KernelSHAP vs GradientSHAP:  mean={np.mean(corr_ks_gs):.3f} ± {np.std(corr_ks_gs):.3f}")
print(f"  KernelSHAP vs DeepLIFT SHAP: mean={np.mean(corr_ks_dl):.3f} ± {np.std(corr_ks_dl):.3f}")
print(f"  GradientSHAP vs DeepLIFT:    mean={np.mean(corr_gs_dl):.3f} ± {np.std(corr_gs_dl):.3f}")
print()
print("Interpretation:")
print("  r > 0.9: very high agreement")
print("  r > 0.7: moderate agreement")
print("  r < 0.5: significant disagreement — investigate further")

## 9. Attribution Heatmap: All Methods vs. All Test Inputs

Visualize the full attribution matrix for each method — which features drive predictions across different inputs.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

all_attrs = [
    (ks_reference.numpy(), 'KernelSHAP (n=500)'),
    (gs_reference.numpy(), 'GradientSHAP (n=50)'),
    (dl_reference.numpy(), 'DeepLIFT SHAP (n=50)'),
]

# Compute shared color scale
all_vals = np.concatenate([a for a, _ in all_attrs])
vmax = np.abs(all_vals).max()

input_labels = [f"P={test_probs[i]:.2f}" for i in test_indices]

for ax, (attrs, title) in zip(axes, all_attrs):
    im = ax.imshow(attrs, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
    ax.set_xticks(range(len(feature_names)))
    ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(n_inputs))
    ax.set_yticklabels(input_labels, fontsize=9)
    ax.set_title(title, fontweight='bold', fontsize=11)
    plt.colorbar(im, ax=ax, shrink=0.8, label='Attribution φᵢ')

plt.suptitle('Attribution Heatmaps: All SHAP Methods\n(Red=positive/toward >50K, Blue=negative/toward ≤50K)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_attribution_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: shap_attribution_heatmaps.png")

## 10. Feature Importance: Per-Method Bar Chart

For the decision-boundary input (near P=0.5), compare feature attributions across all methods.

In [ ]:
# Use the input closest to decision boundary (index 2 in our sorted selection)
boundary_idx = 2

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

all_attrs_boundary = [
    (ks_reference[boundary_idx].numpy(), 'KernelSHAP\n(n=500)', '#e41a1c'),
    (gs_reference[boundary_idx].numpy(), 'GradientSHAP\n(n=50)', '#377eb8'),
    (dl_reference[boundary_idx].numpy(), 'DeepLIFT SHAP\n(n=50)', '#4daf4a'),
]

for ax, (attrs, title, color) in zip(axes, all_attrs_boundary):
    # Sort by absolute value
    sorted_idx_feat = np.argsort(np.abs(attrs))[::-1]
    sorted_attrs = attrs[sorted_idx_feat]
    sorted_names = [feature_names[i] for i in sorted_idx_feat]

    bar_colors = ['#d73027' if v > 0 else '#4575b4' for v in sorted_attrs]
    bars = ax.barh(range(len(sorted_attrs)), sorted_attrs, color=bar_colors)
    ax.set_yticks(range(len(sorted_names)))
    ax.set_yticklabels(sorted_names, fontsize=9)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Attribution (φᵢ)')
    ax.set_title(title, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

prob_val = test_probs[test_indices[boundary_idx]].item()
plt.suptitle(f'Feature Attributions at Decision Boundary\n(Input: P(>50K)={prob_val:.3f})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_boundary_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: shap_boundary_comparison.png")

## 11. KernelSHAP Sample Count: Convergence Analysis

How many samples does KernelSHAP need before attributions stabilize?

In [ ]:
# Compare KernelSHAP at different n_samples vs. the n=500 reference
sample_counts = [50, 100, 200]
reference_attrs = ks_attributions[500].numpy()  # "ground truth"

correlations_by_sample = {}
for n in sample_counts:
    attrs = ks_attributions[n].numpy()
    corrs = [spearmanr(attrs[i], reference_attrs[i]).correlation for i in range(n_inputs)]
    correlations_by_sample[n] = np.mean(corrs)

fig, ax = plt.subplots(figsize=(8, 4))
sample_vals = list(correlations_by_sample.keys())
corr_vals = list(correlations_by_sample.values())
ax.plot(sample_vals, corr_vals, 'o-', color='#e41a1c', linewidth=2, markersize=8)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='Perfect agreement')
ax.axhline(0.9, color='#4daf4a', linestyle='--', alpha=0.7, label='r=0.9 threshold')
ax.set_xlabel('KernelSHAP n_samples')
ax.set_ylabel('Spearman r vs. n=500 reference')
ax.set_title('KernelSHAP Convergence with Sample Count')
ax.legend()
ax.set_ylim(0.7, 1.05)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('kernelshap_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: kernelshap_convergence.png")

## 12. Summary: Method Selection Guide

Based on our benchmarks, here's when to use each method:

In [ ]:
# Print a summary table
import time

# Final comparison table
print("=" * 65)
print("SHAP METHODS COMPARISON SUMMARY")
print("=" * 65)
print(f"{'Method':<20} {'Time (s)':<12} {'Corr. w/ KS-500':<18} {'Best for'}")
print("-" * 65)

ks_corr = "(reference)"
gs_corr = f"{np.mean([spearmanr(gs_reference[i].numpy(), ks_reference[i].numpy()).correlation for i in range(n_inputs)]):.3f}"
dl_corr = f"{np.mean([spearmanr(dl_reference[i].numpy(), ks_reference[i].numpy()).correlation for i in range(n_inputs)]):.3f}"

rows = [
    ("KernelSHAP(200)",  f"{ks_times[200]:.2f}s",  ks_corr, "Black-box models"),
    ("GradientSHAP(50)", f"{gs_times[50]:.2f}s",   gs_corr, "Transformers, custom"),
    ("DeepLIFT SHAP(50)",f"{dl_times[50]:.2f}s",   dl_corr, "Standard CNNs, ReLU"),
]
for name, t, corr, use in rows:
    print(f"{name:<20} {t:<12} {corr:<18} {use}")

print("=" * 65)
print("\nKey findings:")
print("1. All three methods agree on top features (r > 0.85 typically)")
print("2. GradientSHAP and DeepLIFT SHAP are 3-10x faster than KernelSHAP")
print("3. DeepLIFT SHAP has near-zero convergence delta (exact efficiency)")
print("4. KernelSHAP needs n_samples >= 200 for stable estimates")

## Self-Check Questions

Before moving on, answer these questions to test your understanding:

1. **Convergence:** Why does DeepLIFT SHAP's convergence delta approach machine precision, while GradientSHAP's delta is only small but not negligible?

2. **Background size:** What happens to GradientSHAP attributions if you use only 1 background sample instead of 50? Would the attributions be more like GradientSHAP or like Integrated Gradients?

3. **Method selection:** A colleague is running KernelSHAP on a ResNet-50 image classifier and it's taking 45 minutes per image. Which method would you recommend instead, and why?

4. **Correlation interpretation:** Two methods give Spearman r=0.7 for the same input. Is this good or bad? What would you investigate?

**Exercise:** Modify the code above to add a zero baseline to the comparison. Run DeepLIFT (not DeepLIFT SHAP) with a zero baseline and add its attributions to the bar chart in Section 10. How does single-baseline DeepLIFT compare to DeepLIFT SHAP with 50 backgrounds?